# Phase 3 — Dimensionality Reduction v2 (DimReducer)

A pipeline-ready demonstration of PCA, ICA, and LDA on trial-averaged
neural spike data. Uses `DimReducer` from `decoding.dim_reduction` as the
core abstraction, alongside feature extraction from `decoding.feature_extraction`.

| Section | Content |
|---|---|
| Imports | All packages |
| S3 Connection | AWS credentials + DatasetLoader |
| Data Loading | DANDI_00070 session summary |
| 1 — Feature Extraction | compute_trial_averages + direction labels |
| 2 — Visualization | SessionPlots.pca() neural trajectory |
| 3 — PCA | DimReducer + scree plot + scatter |
| 4 — ICA | DimReducer + IC scatter |
| 5 — LDA | DimReducer + LD scatter |
| 6 — Comparison | Silhouette + reconstruction error table |

In [ ]:
from pathlib import Path
import sys
_repo_root = Path.cwd() if (Path.cwd() / "decoding").is_dir() else Path.cwd().parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import os
import configparser

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score

from bci_decoding_dataset import DatasetLoader
from bci_decoding_dataset.neuroviz import SessionPlots

from decoding import DimReducer #de dim_reduction.py


print("\u2713 All imports successful")

✓ All imports successful


In [2]:
credentials_path = os.path.expanduser("~/.aws/credentials")
config = configparser.ConfigParser()
config.read(credentials_path)
profile = os.environ.get("AWS_PROFILE", "cv-pc")

loader = DatasetLoader(
    aws_store=True,
    s3_bucket="solzbacher-lab-motor-decoding-ds",
    s3_key="datasets/Combined_Motor_Datasets",
    aws_access_key_id=config[profile]["aws_access_key_id"],
    aws_secret_access_key=config[profile]["aws_secret_access_key"],
)
print("\u2713 Connected to S3")

✓ Successfully opened Zarr from S3: s3://solzbacher-lab-motor-decoding-ds/datasets/Combined_Motor_Datasets
✓ Connected to S3


In [6]:
import os
os.environ["AWS_PROFILE"] = "cv-pc"

loader = DatasetLoader(
    aws_store=True,
    s3_bucket="solzbacher-lab-motor-decoding-ds",
    s3_key="datasets/Combined_Motor_Datasets",
    # sin credenciales — s3fs las toma del entorno automáticamente
)

✓ Successfully opened Zarr from S3: s3://solzbacher-lab-motor-decoding-ds/datasets/Combined_Motor_Datasets


In [8]:
# DANDI_00070: ~3000 trials, ~191 electrodes — best for discrete decoding
sessions = loader.filter_sessions("dataset_id", "DANDI_00070")
session_id = sessions[0]

ds = loader.get_processed_data_from_session(session_id)
print(f"Session:       {session_id}")
print(f"Subject:       {ds.attrs.get('subject_id', 'unknown')}")
print(f"Task:          {ds.attrs.get('task_type', 'unknown')}")
print(f"Sampling rate: {ds.attrs.get('sampling_rate', 'unknown')} Hz")
print(f"Spikes shape:  {ds['spikes'].shape}  (n_electrodes \u00d7 n_time)")
print(f"Duration:      {ds['spikes'].shape[1] / ds.attrs.get('sampling_rate', 'unknown'):.0f} s")

Reading data from session: 20090812
Session:       20090812
Subject:       unknown
Task:          center_out
Sampling rate: 1000.0 Hz
Spikes shape:  (191, 14528754)  (n_electrodes × n_time)
Duration:      14529 s


## Section 1 — Feature Extraction

`compute_trial_averages` bins spike trains at 50 ms resolution, then
averages each electrode over the reach phase (trial_phase == 2) of every
successful trial, yielding one feature vector per trial.  This collapses
the time axis and gives us a clean `(n_trials, n_electrodes)` matrix that
is the correct input for dimensionality reduction.

`compute_direction_labels` assigns a discrete reach-direction index (0–7)
to each trial using mean velocity during the reach phase.

In [9]:
X_trials, valid_trial_ids = compute_trial_averages(ds, bin_size_ms=50)

# Align direction labels to the subset of trials returned by compute_trial_averages
all_direction_labels = compute_direction_labels_from_position(ds)
all_trial_ids = np.unique(ds["trial_id"].values[ds["trial_id"].values > 0])
tid_to_label = dict(zip(all_trial_ids.tolist(), all_direction_labels.tolist()))
direction_labels = np.array([tid_to_label[int(tid)] for tid in valid_trial_ids])

# Drop any trials whose reach phase had no velocity samples (label == -1)
valid_mask = direction_labels != -1
X_trials = X_trials[valid_mask]
direction_labels = direction_labels[valid_mask]

print(f"X_trials shape:   {X_trials.shape}  (n_trials \u00d7 n_electrodes)")
print(f"Directions:       {np.unique(direction_labels).tolist()}  ({len(np.unique(direction_labels))} classes)")

NameError: name 'compute_trial_averages' is not defined

## Section 2 — Visualization with SessionPlots

`SessionPlots.pca()` from the `neuroviz` package computes a smoothed PCA
trajectory and returns a Plotly figure — it is **visualization only** and does
not expose the underlying components.  We use it here for a quick sanity check
of the neural trajectory structure before running our own `DimReducer` pipeline.

In [ ]:
sp = SessionPlots(loader, session_id)
sp.pca(t_start=0, t_end=30, n_start=0, n_end=49, mode='continuous').show()

## Section 3 — PCA with DimReducer

PCA finds orthogonal axes of maximum variance in the trial-averaged spike
counts.  It is the natural first step: it is fast, interpretable, and the
scree plot tells us how many components are worth retaining.  If a few PCs
capture most variance, the population code is low-dimensional — a prerequisite
for efficient decoding.

In [ ]:
reducer_pca = DimReducer(method='pca', n_components=10)
X_pca = reducer_pca.fit_transform(X_trials)
print(f"PCA output shape: {X_pca.shape}")

In [ ]:
evr = reducer_pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(evr) + 1), evr * 100, color='steelblue', label='Per component')
ax.plot(range(1, len(evr) + 1), np.cumsum(evr) * 100,
        'o-', color='tomato', label='Cumulative')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance (%)')
ax.set_title('PCA Scree Plot')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                c=direction_labels, cmap='tab10', s=20, alpha=0.7)
fig.colorbar(sc, ax=ax, label='Reach direction (0\u20137)')
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('PCA \u2014 PC1 vs PC2 colored by reach direction')
plt.tight_layout()
plt.show()

## Section 4 — ICA with DimReducer

FastICA seeks statistically independent (maximally non-Gaussian) components.
Where PCA maximizes variance, ICA maximizes statistical independence — useful
for separating mixed neural sources that contribute non-Gaussian firing
patterns (bursts, silences).  Components have no natural ordering, so we
compare IC1 vs IC2 directly with the PCA scatter.

In [ ]:
reducer_ica = DimReducer(method='ica', n_components=10)
X_ica = reducer_ica.fit_transform(X_trials)
print(f"ICA output shape: {X_ica.shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(X_ica[:, 0], X_ica[:, 1],
                c=direction_labels, cmap='tab10', s=20, alpha=0.7)
fig.colorbar(sc, ax=ax, label='Reach direction (0\u20137)')
ax.set_xlabel('IC 1')
ax.set_ylabel('IC 2')
ax.set_title('ICA \u2014 IC1 vs IC2 colored by reach direction')
plt.tight_layout()
plt.show()

## Section 5 — LDA with DimReducer

Linear Discriminant Analysis is **supervised**: it uses direction labels
to find axes that maximize between-class variance relative to within-class
variance.  Unlike PCA and ICA, LDA directly optimizes class separability, so
we expect the clearest clusters here.  The maximum number of LDA components
is n_classes \u2212 1 = 7 for 8 directions; `DimReducer` applies this cap
automatically.

> **Note on component count:** LDA is capped at *n\_classes − 1* components.
> If the session contains fewer than 8 unique direction labels (e.g. some
> directions were never reached in this recording), `DimReducer` automatically
> reduces the component count. With only 2 classes present, LDA produces
> 1 component. The plot below adapts accordingly.

In [ ]:
reducer_lda = DimReducer(method='lda', n_components=7)
X_lda = reducer_lda.fit_transform(X_trials, y=direction_labels)
print(f"LDA output shape: {X_lda.shape}")

In [ ]:
if X_lda.shape[1] >= 2:
    fig, ax = plt.subplots(figsize=(6, 5))
    sc = ax.scatter(X_lda[:, 0], X_lda[:, 1],
                    c=direction_labels, cmap='tab10', s=20, alpha=0.7)
    fig.colorbar(sc, ax=ax, label='Reach direction (0–7)')
    ax.set_xlabel('LD 1')
    ax.set_ylabel('LD 2')
    ax.set_title('LDA — LD1 vs LD2 colored by reach direction')
else:
    # Only 1 LDA component available: fall back to overlapping 1-D histograms
    fig, ax = plt.subplots(figsize=(8, 4))
    for d in np.unique(direction_labels):
        ax.hist(X_lda[direction_labels == d, 0], bins=30, alpha=0.5, label=f'Dir {d}')
    ax.set_xlabel('LD 1')
    ax.set_ylabel('Count')
    ax.set_title(
        f'LDA — LD1 histogram by direction '
        f'(only {X_lda.shape[1]} component available)'
    )
    ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# silhouette usando solo PC1 y PC2
from sklearn.metrics import silhouette_score
print(silhouette_score(X_pca[:, :2], direction_labels))

# silhouette usando solo PC1, PC2, PC3
print(silhouette_score(X_pca[:, :3], direction_labels))

## Section 6 — Method Comparison

Two metrics summarize how well each method works for this data:

- **Silhouette score** (\u22121 to 1): cluster quality of reach directions in
  the reduced space.  Higher = better separation.  LDA should win because it
  directly maximizes this separation.
- **Reconstruction error** (MSE on StandardScaler-scaled data): how faithfully
  the reduced representation reconstructs the original features.  Available for
  PCA and ICA (both have `inverse_transform`); LDA is discriminative-only.
  PCA should have the lowest error (it minimizes this quantity by construction).

In [ ]:
# --- Shape verification ---
print(f'X_pca: {X_pca.shape}  X_ica: {X_ica.shape}  X_lda: {X_lda.shape}')
print(f'direction_labels: {direction_labels.shape}  unique: {np.unique(direction_labels)}')
assert X_pca.shape[0] == len(direction_labels), (
    f'Shape mismatch: X_pca rows {X_pca.shape[0]} vs labels {len(direction_labels)}'
)

# --- Reconstruction error in original (unscaled) space ---
# Use already-fitted X_pca / X_ica; inverse through model then scaler
X_rec_pca = reducer_pca.scaler_.inverse_transform(
    reducer_pca.model_.inverse_transform(X_pca)
)
rec_pca = float(np.mean((X_trials - X_rec_pca) ** 2))

X_rec_ica = reducer_ica.scaler_.inverse_transform(
    reducer_ica.model_.inverse_transform(X_ica)
)
rec_ica = float(np.mean((X_trials - X_rec_ica) ** 2))

if rec_pca == rec_ica:
    print(f'DEBUG: rec_pca == rec_ica == {rec_pca}')
    print(f'  X_rec_pca[0, :5]: {X_rec_pca[0, :5]}')
    print(f'  X_rec_ica[0, :5]: {X_rec_ica[0, :5]}')
    print(f'  X_pca[0, :5]:     {X_pca[0, :5]}')
    print(f'  X_ica[0, :5]:     {X_ica[0, :5]}')
else:
    print(f'rec_pca={rec_pca:.6f}  rec_ica={rec_ica:.6f}')

# --- Silhouette scores ---
sil_pca = silhouette_score(X_pca, direction_labels)
sil_ica = silhouette_score(X_ica, direction_labels)
sil_lda = silhouette_score(X_lda, direction_labels)
print(f'sil_pca={sil_pca:.4f}  sil_ica={sil_ica:.4f}  sil_lda={sil_lda:.4f}')

# --- Comparison table ---
evr_pca_pct = reducer_pca.explained_variance_ratio_.sum() * 100

comparison = pd.DataFrame({
    "Method": [
        f"PCA ({X_pca.shape[1]} components)",
        f"ICA ({X_ica.shape[1]} components)",
        f"LDA ({X_lda.shape[1]} components)",
    ],
    "Silhouette Score": [
        f"{sil_pca:.4f}",
        f"{sil_ica:.4f}",
        f"{sil_lda:.4f}",
    ],
    "Reconstruction Error": [
        f"{rec_pca:.6f}",
        f"{rec_ica:.6f}",
        "N/A",
    ],
    "Notes": [
        f"Explains {evr_pca_pct:.1f}% variance",
        "No variance ordering; captures non-Gaussian structure",
        "Supervised — uses direction labels; max n_classes−1 components",
    ],
})
display(comparison)

In [ ]:
# silhouette usando solo PC1 y PC2
from sklearn.metrics import silhouette_score
print(silhouette_score(X_pca[:, :2], direction_labels))

# silhouette usando solo PC1, PC2, PC3
print(silhouette_score(X_pca[:, :3], direction_labels))